# CareSignal on a Colab GPU

Runtime → Change runtime type → **T4 GPU**, then run top to bottom (~45 min).

1. RoBERTa fine-tuning on GPU (fp16), same setup as the CPU run, then a 50k-sentence run.
2. LLM serving benchmark: **vLLM vs Hugging Face `generate`** on the same prompts.

Results land in `reports/`; the last cell zips them for download.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists('caresignal'):
    !git clone https://github.com/vedantjhunjhunwal/caresignal.git
# If the repo isn't public yet: upload caresignal.zip in the Files panel and run  !unzip -q caresignal.zip
%cd caresignal
!pip install -q -e .
!bash scripts/get_data.sh

## 1. Fine-tune RoBERTa-base on the GPU
Same data, epochs and dev-based model selection as the CPU run (66 min on 2 CPU cores, macro-F1 0.789).

In [ ]:
%env MLFLOW_TRACKING_URI=sqlite:///mlflow.db
!python -m caresignal.nlp.finetune --model models/roberta-base --train-n 8000 --epochs 2 --bs 32 --out models/rct-roberta-gpu --report reports/nlp_roberta_gpu_8k.json

In [ ]:
!python -m caresignal.nlp.finetune --model models/roberta-base --train-n 50000 --epochs 2 --bs 64 --out models/rct-roberta-gpu-50k --report reports/nlp_roberta_gpu_50k.json

## 2. LLM serving: vLLM vs Hugging Face generate
Installing vLLM can change the torch version; if pip asks, use Runtime → Restart session, then run the next two cells again.

In [ ]:
!pip install -q vllm

In [ ]:
%cd /content/caresignal
%env MLFLOW_TRACKING_URI=sqlite:///mlflow.db
!python scripts/vllm_bench.py --model Qwen/Qwen2.5-1.5B-Instruct --n 200

## 3. Results

In [ ]:
import json, glob
for f in sorted(glob.glob('reports/nlp_roberta_gpu_*.json')):
    r = json.load(open(f)); print(f, r['gpu'], r['n_train'], 'minutes', r['minutes'], 'macro-F1', round(r['test']['macro avg']['f1-score'], 4))
b = json.load(open('reports/vllm_bench.json'))
print(b['gpu'], 'HF', round(b['hf']['tokens_per_s']), 'tok/s | vLLM', round(b['vllm']['tokens_per_s']), 'tok/s | speedup', round(b['throughput_speedup'], 1), 'x')
!zip -q -r gpu_results.zip reports mlflow.db
from google.colab import files; files.download('gpu_results.zip')